# Demand Driver Analysis

This notebook measures how much price, weather, holidays and school breaks,
marketing, and calendar structure contribute to out-of-sample demand
accuracy. Each feature group is removed from the selected SARIMAX model and
re-evaluated on the same 12 forecast origins and 360 target dates.

A positive MAE increase means the removed group contributed useful predictive
information. These results describe predictive association, not causal effects.

## Load the Validated Forecast Contract

The analysis reads the same PostgreSQL view and persisted SARIMAX backtests
used by the production demand pipeline.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sqlalchemy import create_engine
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.statespace.sarimax import SARIMAX

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.forecast import (
    CATEGORICAL_FEATURES,
    FORECAST_HORIZON,
    SARIMAX_NUMERIC_FEATURES,
    build_backtest_fold,
    build_backtest_plan,
    build_final_forecast_data,
    calculate_forecast_metrics,
    load_forecast_input,
    load_latest_sarimax_backtests,
    prepare_model_features,
    validate_forecast_input,
)

DATABASE_URL = "postgresql+psycopg:///orlando_demand_revenue"
engine = create_engine(DATABASE_URL)

forecast_input = load_forecast_input(engine)
validate_forecast_input(forecast_input)

historical = forecast_input.loc[
    forecast_input["target_demand"].notna()
].copy()

assert len(historical) == 1096
assert forecast_input["target_demand"].isna().sum() == 30

historical[["calendar_date", "target_demand"]].agg(
    ["min", "max", "count"]
)

## Shared Backtest and Feature Groups

The persisted full-model predictions are the control. Every ablation uses
the same expanding training windows, leakage-safe monthly weather
climatology, SARIMAX order, and 30-day evaluation horizons.

In [ ]:
backtest_plan = build_backtest_plan(
    forecast_input,
    start="2024-12-31",
    end="2025-11-30",
    forecast_horizon=FORECAST_HORIZON,
)

FEATURE_GROUPS = {
    "calendar_structure": {
        "numeric": ["trend_days"],
        "categorical": ["day_of_week", "month_number"],
    },
    "holiday_school_break": {
        "numeric": ["holiday_flag", "school_break_flag"],
        "categorical": [],
    },
    "planned_price": {
        "numeric": ["planned_price_multiplier"],
        "categorical": [],
    },
    "marketing": {
        "numeric": [
            "maximum_planned_discount",
            "paid_media_flag",
            "email_campaign_flag",
            "bundle_campaign_flag",
            "discount_campaign_flag",
        ],
        "categorical": [],
    },
    "weather": {
        "numeric": [
            "model_temperature_f",
            "model_precipitation_in",
            "model_weather_risk",
        ],
        "categorical": [],
    },
}

grouped_numeric = {
    feature
    for group in FEATURE_GROUPS.values()
    for feature in group["numeric"]
}
grouped_categorical = {
    feature
    for group in FEATURE_GROUPS.values()
    for feature in group["categorical"]
}

assert grouped_numeric == set(SARIMAX_NUMERIC_FEATURES)
assert grouped_categorical == set(CATEGORICAL_FEATURES)
assert len(backtest_plan) == 12

backtest_plan

In [ ]:
# Purpose:
# Build a fold-specific transformer after one feature group is removed.
#
# Used by:
# Every grouped SARIMAX ablation specification.
def make_ablation_preprocessor(
    numeric_features,
    categorical_features,
):
    transformers = []

    if numeric_features:
        transformers.append(
            (
                "numeric",
                StandardScaler(),
                list(numeric_features),
            )
        )

    if categorical_features:
        transformers.append(
            (
                "categorical",
                OneHotEncoder(
                    drop="first",
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                list(categorical_features),
            )
        )

    if not transformers:
        raise ValueError("At least one feature must remain.")

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )


# Purpose:
# Fit one leakage-safe ablated SARIMAX fold and return aligned predictions.
#
# Used by:
# The grouped ablation backtest loop.
def fit_ablation_fold(
    data,
    cutoff,
    model_spec,
    numeric_features,
    categorical_features,
):
    training, validation, _ = build_backtest_fold(
        data,
        cutoff,
        forecast_horizon=FORECAST_HORIZON,
    )

    series_start = data["calendar_date"].min()
    training_features = prepare_model_features(
        training.sort_values("calendar_date"),
        series_start,
    )
    validation_features = prepare_model_features(
        validation.sort_values("calendar_date"),
        series_start,
    )

    preprocessor = make_ablation_preprocessor(
        numeric_features,
        categorical_features,
    )

    training_exog = np.asarray(
        preprocessor.fit_transform(training_features),
        dtype=float,
    )
    validation_exog = np.asarray(
        preprocessor.transform(validation_features),
        dtype=float,
    )

    model = SARIMAX(
        endog=training_features["target_demand"].astype(float),
        exog=training_exog,
        order=(1, 0, 1),
        seasonal_order=(1, 0, 0, 7),
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )

    fitted = None
    optimizer_used = None
    convergence_warnings = 0

    # Optimizers estimate the same model specification. The fallback
    # prevents one difficult ablation fold from invalidating the study.
    for optimizer, max_iterations in (("lbfgs", 500), ("powell", 500)):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            candidate = model.fit(
                method=optimizer,
                maxiter=max_iterations,
                disp=False,
            )

        convergence_warnings += sum(
            issubclass(item.category, ConvergenceWarning)
            for item in caught
        )
        fitted = candidate
        optimizer_used = optimizer
        if bool(candidate.mle_retvals.get("converged", False)):
            break

    converged = bool(fitted.mle_retvals.get("converged", False))

    if not converged:
        raise RuntimeError(
            f"{model_spec} did not converge for {cutoff}."
        )

    predicted = np.maximum(
        np.asarray(
            fitted.get_forecast(
                steps=len(validation_features),
                exog=validation_exog,
            ).predicted_mean
        ),
        0,
    )

    return pd.DataFrame({
        "forecast_created_date": pd.Timestamp(cutoff),
        "target_date": validation_features["calendar_date"].to_numpy(),
        "horizon_days": (
            validation_features["calendar_date"]
            - pd.Timestamp(cutoff)
        ).dt.days.to_numpy(),
        "model_spec": model_spec,
        "predicted_demand": predicted,
        "actual_demand": validation_features[
            "target_demand"
        ].astype(float).to_numpy(),
        "converged": converged,
        "convergence_warnings": convergence_warnings,
        "optimizer_used": optimizer_used,
    })

## Run the Grouped Ablation Backtest

This cell fits 60 SARIMAX folds: five feature-group removals across 12
origins. The full-model control is loaded from the immutable production
backtest table rather than refitted.

In [ ]:
full_predictions = load_latest_sarimax_backtests(engine).copy()
full_predictions["model_spec"] = "full_model"
full_predictions["converged"] = True
full_predictions["convergence_warnings"] = 0
full_predictions["optimizer_used"] = "persisted_lbfgs"

ablation_folds = [full_predictions]

for group_name, removed_features in FEATURE_GROUPS.items():
    numeric_features = [
        feature
        for feature in SARIMAX_NUMERIC_FEATURES
        if feature not in removed_features["numeric"]
    ]
    categorical_features = [
        feature
        for feature in CATEGORICAL_FEATURES
        if feature not in removed_features["categorical"]
    ]
    model_spec = f"without_{group_name}"

    for fold_number, cutoff in enumerate(
        backtest_plan["forecast_created_date"],
        start=1,
    ):
        ablation_folds.append(
            fit_ablation_fold(
                forecast_input,
                cutoff,
                model_spec,
                numeric_features,
                categorical_features,
            )
        )

    print(f"Completed {model_spec}: 12/12 folds")

driver_predictions = pd.concat(
    ablation_folds,
    ignore_index=True,
)

expected_specs = 1 + len(FEATURE_GROUPS)
assert driver_predictions["model_spec"].nunique() == expected_specs
assert driver_predictions.groupby("model_spec").size().eq(360).all()
assert driver_predictions["converged"].all()

driver_predictions.groupby("model_spec").agg(
    observations=("actual_demand", "size"),
    converged_folds=("forecast_created_date", "nunique"),
    convergence_warnings=("convergence_warnings", "sum"),
    optimizers=(
        "optimizer_used",
        lambda values: ", ".join(sorted(values.unique())),
    ),
)

In [ ]:
driver_model_metrics = (
    driver_predictions.groupby("model_spec")
    .apply(
        calculate_forecast_metrics,
        include_groups=False,
    )
    .sort_values("mae")
)

full_model_mae = driver_model_metrics.loc[
    "full_model",
    "mae",
]

driver_importance = (
    driver_model_metrics.drop(index="full_model")
    .reset_index()
    .rename(columns={"model_spec": "ablation_model"})
)

driver_importance["feature_group"] = (
    driver_importance["ablation_model"]
    .str.removeprefix("without_")
)
driver_importance["mae_increase"] = (
    driver_importance["mae"] - full_model_mae
)
driver_importance["mae_increase_pct"] = (
    driver_importance["mae_increase"]
    / full_model_mae
    * 100
)
driver_importance["predictive_contribution"] = np.where(
    driver_importance["mae_increase"].gt(0),
    "improves_accuracy",
    "no_incremental_gain",
)

driver_importance = driver_importance.sort_values(
    "mae_increase",
    ascending=False,
).set_index("feature_group")

driver_importance[
    [
        "observations",
        "mae",
        "rmse",
        "mape_pct",
        "forecast_bias_pct",
        "mae_increase",
        "mae_increase_pct",
        "predictive_contribution",
    ]
].round(2)

## Direction of Association

Ablation ranks feature groups by out-of-sample contribution. A regularized
Ridge model with time-series cross-validation is used only to provide stable
directional context when correlated business features make raw SARIMAX
coefficients unstable. Numeric coefficients represent a one-standard-deviation
feature change; categorical coefficients are relative to the omitted category.

In [ ]:
full_training, _, _ = build_final_forecast_data(forecast_input)
series_start = forecast_input["calendar_date"].min()
full_training_features = prepare_model_features(
    full_training,
    series_start,
)

direction_preprocessor = make_ablation_preprocessor(
    list(SARIMAX_NUMERIC_FEATURES),
    list(CATEGORICAL_FEATURES),
)
direction_exog = direction_preprocessor.fit_transform(
    full_training_features
)
direction_feature_names = (
    direction_preprocessor.get_feature_names_out()
)

direction_model = RidgeCV(
    alphas=np.logspace(-3, 4, 29),
    cv=TimeSeriesSplit(n_splits=6),
    scoring="neg_mean_absolute_error",
).fit(
    direction_exog,
    full_training_features["target_demand"].astype(float),
)

coefficient_summary = pd.DataFrame({
    "feature": direction_feature_names,
    "ridge_coefficient": direction_model.coef_,
})


# Purpose:
# Map transformed coefficient names back to their business feature group.
#
# Used by:
# The coefficient interpretation table.
def identify_feature_group(feature_name):
    for group_name, group_features in FEATURE_GROUPS.items():
        candidates = [
            *group_features["numeric"],
            *group_features["categorical"],
        ]
        if any(
            feature_name == candidate
            or feature_name.startswith(f"{candidate}_")
            for candidate in candidates
        ):
            return group_name

    raise ValueError(f"Unmapped feature: {feature_name}")


coefficient_summary["feature_group"] = coefficient_summary[
    "feature"
].map(identify_feature_group)
coefficient_summary["association_direction"] = np.where(
    coefficient_summary["ridge_coefficient"].ge(0),
    "positive",
    "negative",
)
coefficient_summary["absolute_coefficient"] = coefficient_summary[
    "ridge_coefficient"
].abs()

print(f"Selected Ridge alpha: {direction_model.alpha_:.4f}")

coefficient_summary.sort_values(
    ["feature_group", "absolute_coefficient"],
    ascending=[True, False],
)[
    [
        "feature_group",
        "feature",
        "ridge_coefficient",
        "association_direction",
    ]
].round(4)

## Interpretation Contract

Use the ablation table to rank out-of-sample predictive contribution. Use the
regularized coefficients only to describe conditional direction; Ridge does not
produce the confidence intervals or p-values required for inference. Do not
interpret either result as causal price elasticity or campaign impact because
price and marketing were not randomly assigned. The separate campaign analysis
will estimate campaign-associated lift with explicit controls and uncertainty.